# 1. Limpieza y preparación de datos

**Objetivo:** leer el archivo `datos/datalog.txt`, extraer las mediciones de temperatura y humedad de los sensores Casa Guadua, Casa Plástico y Ambiente, limpiar valores erróneos y asignar un tiempo relativo en minutos.

In [ ]:
from pathlib import Path

import re
import pandas as pd
import numpy as np

ruta_raw = Path('datos/datalog.txt')

## 1.2 Leer el archivo raw


In [ ]:
with ruta_raw.open('r', encoding='utf-8') as f:
    lineas = f.readlines()

## 1.3 Parsear las líneas para extraer cada muestra.


In [ ]:
muestras = []
muestra_actual = {}
for linea in lineas:
    linea = linea.strip()
    if not linea or linea.startswith('===') or linea.startswith('Formato:'):
        continue
    if linea.startswith("Muestra #"):
        if muestra_actual:
            muestras.append(muestra_actual)
            muestra_actual = {}
        num = int(re.search(r"\d+", linea).group())
        muestra_actual["muestra"] = num
    elif linea.startswith("S1 Casa Guadua:"):
        valores = re.findall(r'(-?\d+(?:\.\d+)?)', linea.split(':', 1)[1])
        if len(valores) >= 2:
            muestra_actual["T_guadua"] = float(valores[0])
            muestra_actual["H_guadua"] = float(valores[1])
    elif linea.startswith("S2 Casa Plastico:"):
        valores = re.findall(r'(-?\d+(?:\.\d+)?)', linea.split(':', 1)[1])
        if len(valores) >= 2:
            muestra_actual["T_plastico"] = float(valores[0])
            muestra_actual["H_plastico"] = float(valores[1])
    elif linea.startswith("S4 Ambiente :"):
        valores = re.findall(r'(-?\d+(?:\.\d+)?)', linea.split(':', 1)[1])
        if len(valores) >= 2:
            muestra_actual["T_ambiente"] = float(valores[0])
            muestra_actual["H_ambiente"] = float(valores[1])
    elif linea.startswith("--------------------------------"):
        if muestra_actual:
            muestras.append(muestra_actual)
            muestra_actual = {}

if muestra_actual:
    muestras.append(muestra_actual)

## 1.4 Crear DataFrame y limpiar


In [ ]:
df = pd.DataFrame(muestras)
df_clean = df.replace(-999, np.nan).dropna().copy()
df_clean['tiempo_min'] = df_clean['muestra'] - 1
columnas = ['muestra', 'tiempo_min', 'T_guadua', 'H_guadua', 'T_plastico', 'H_plastico', 'T_ambiente', 'H_ambiente']
df_clean = df_clean[columnas].reset_index(drop=True)